In [1]:
from pdstools import TreeAnalysis
import polars as pl

### set folder and file paths

In [2]:
folder_path = "/Users/hasss/dev/bol.com/modelSnapshots/"
model_filename = "Data-Decision-ADM-ModelSnapshot_AdmModelsSnapshot_20230308T113755_GMT/data.json"
plots_folder_path = "/Users/hasss/dev/pega-datascientist-tools/output"
tree_analysis = TreeAnalysis(
    past_n_snapshots='all', 
    compare_first_n_trees=4, 
    folder_path=folder_path,
    model_filename=model_filename,
    plots_folder_path=plots_folder_path
)

In [3]:
df = tree_analysis.get_df()

AGB models found: shape: (2, 1)
┌───────────────────────────────┐
│ Configuration                 │
│ ---                           │
│ cat                           │
╞═══════════════════════════════╡
│ Mobile_Click_Through_Rate_AGB │
│ Web_Click_Through_Rate_AGB    │
└───────────────────────────────┘


100%|██████████| 13/13 [00:08<00:00,  1.58it/s]


Processing for channel: Mobile_Click_Through_Rate_AGB
Comparing snapshots 2023-03-03 06:00:01 and 2023-03-04 06:00:01
Comparing snapshots 2023-03-04 06:00:01 and 2023-03-05 06:00:01
Comparing snapshots 2023-03-05 06:00:01 and 2023-03-06 06:00:01
Comparing snapshots 2023-03-06 06:00:01 and 2023-03-07 06:00:01
Comparing snapshots 2023-03-07 06:00:01 and 2023-03-08 06:00:01
Processing for channel: Web_Click_Through_Rate_AGB
Comparing snapshots 2023-03-03 06:00:01 and 2023-03-04 06:00:01
Comparing snapshots 2023-03-04 06:00:01 and 2023-03-05 06:00:01
Comparing snapshots 2023-03-05 06:00:01 and 2023-03-06 06:00:01
Comparing snapshots 2023-03-06 06:00:01 and 2023-03-07 06:00:01
Comparing snapshots 2023-03-07 06:00:01 and 2023-03-08 06:00:01


In [7]:
df.head().sort(["channel", "snapshot_from", "tree_id"])

channel,snapshot_from,snapshot_to,snapshot,tree_id,change_type,node,predictor,score,parent_node,gain,split,left_child,right_child,depth,flag
str,str,str,str,str,str,str,str,f64,f64,f64,str,f64,f64,i64,str
"""Mobile_Click_T…","""20230303""","""20230304""","""20230303""","""tree1""","""split_to_split…","""node1""","""IH.Mobile.Inbo…",-0.195688,null,330.958696,"""IH.Mobile.Inbo…",2.0,7.0,1,"""goldenrod"""
"""Mobile_Click_T…","""20230303""","""20230304""","""20230303""","""tree1""","""split_to_split…","""node2""","""Param.DayPartB…",-0.186647,null,107.764421,"""Param.DayPartB…",2.0,5.0,1,"""goldenrod"""
"""Mobile_Click_T…","""20230303""","""20230304""","""20230303""","""tree4""","""identical""","""node1""","""Customer.Behav…",-0.15401,null,88.370225,"""Customer.Behav…",2.0,21.0,1,"""white"""
"""Mobile_Click_T…","""20230303""","""20230304""","""20230303""","""tree4""","""identical""","""node2""","""Customer.Behav…",-0.145737,null,423.912891,"""Customer.Behav…",2.0,37.0,1,"""white"""
"""Mobile_Click_T…","""20230303""","""20230304""","""20230303""","""tree4""","""identical""","""node1""","""IH.Mobile.Inbo…",-0.154388,1.0,248.913296,"""IH.Mobile.Inbo…",3.0,4.0,2,"""white"""


In [11]:
from enum import Enum
CHANGE_TYPES = Enum('ChangeType', ['split_to_prune', 'split_to_split', 'prune_to_split', 'identical'])
print(CHANGE_TYPES.split_to_prune.name)

NODES = Enum('Nodes', ['node1', 'node2'])
print(NODES.node1.name)

split_to_prune
node1


In [12]:
def run_query(query):
    with pl.Config() as cfg:
        cfg.set_fmt_str_lengths(100)
        display(query)

In [13]:

query = (
    df
    .filter(
        (pl.col("change_type") == CHANGE_TYPES.split_to_prune.name) &
        (pl.col("node") == NODES.node1.name)
    )
    .groupby(['channel', 'change_type', 'predictor'])
    .agg(pl.count())
    .select(pl.col('channel', 'change_type', 'predictor', 'count').sort_by("channel", "predictor"))
)
run_query(query)

channel,change_type,predictor,count
str,str,str,u32
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.BehavioralData.DaysSinceLastVisit""",1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.Consent.HasService""",1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.FavouriteConsole""",1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.Is11306CategoryOrderedIn24M""",1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.Is18200CategoryOrderedIn24M""",1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.NrOrdersLast12Months""",1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.NumberOfItemsPurchased""",1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.OrderShareL1Baby""",1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.OrderShareL1KantoorSchool""",1


In [14]:
item = query[13,:]
display(item)

query1 = (
    df.filter(
            (pl.col("node") == NODES.node1.name) &
            (pl.col("channel") == item.select("channel")) &
            (pl.col("change_type") == item.select("change_type")) & 
            (pl.col("predictor") == item.select("predictor"))
        )
)
run_query(query1)

channel,change_type,predictor,count
str,str,str,u32
"""Mobile_Click_T…","""split_to_prune…","""IH.Mobile.Inbo…",1


channel,snapshot_from,snapshot_to,snapshot,tree_id,change_type,node,predictor,score,parent_node,gain,split,left_child,right_child,depth,flag
str,str,str,str,str,str,str,str,f64,f64,f64,str,f64,f64,i64,str
"""Mobile_Click_Through_Rate_AGB""","""20230305""","""20230306""","""20230305""","""tree4""","""split_to_prune""","""node1""","""IH.Mobile.Inbound.Impression.pyHistoricalOutcomeCount""",-0.145275,4.0,0.157733,"""IH.Mobile.Inbound.Impression.pyHistoricalOutcomeCount < 14.0""",11.0,12.0,5,"""crimson"""


In [25]:
query2 = (
    df
    .filter(
        (pl.col("node") == NODES.node1.name) &
        (pl.col("change_type") != "identical")
    )
    .groupby(['channel', 'change_type', 'predictor', 'depth'])
    .agg(pl.count())
)
run_query(query2)

channel,change_type,predictor,depth,count
str,str,str,i64,u32
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""IH.Mobile.Inbound.Clicked.pxLastOutcomeTime.DaysSince""",4,1
"""Web_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.BehavioralData.L1ShelfVisitMostInSession""",4,1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.BehavioralData.DaysSinceLastVisit""",4,1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""IH.Mobile.Inbound.Impression.pyHistoricalOutcomeCount""",5,1
"""Web_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.BehavioralData.VisitShareL1Beauty""",5,1
"""Web_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.BehavioralData.VisitShareL1Games""",5,1
"""Web_Click_Through_Rate_AGB""","""prune_to_split""",null,4,15
"""Web_Click_Through_Rate_AGB""","""split_to_split""","""Customer.DaysSinceLastOrder""",4,1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.Is18200CategoryOrderedIn24M""",5,1


#### query:: average gain of splits that were pruned

In [16]:

query3 = (
    df
    .filter(
        (pl.col("change_type") == CHANGE_TYPES.split_to_prune.name) &
        (pl.col("node") == NODES.node1.name)
    )
    .groupby(['channel'])
    .agg(pl.col("gain").mean())
)
run_query(query3)

channel,gain
str,f64
"""Web_Click_Through_Rate_AGB""",11.412645
"""Mobile_Click_Through_Rate_AGB""",3.323169


#### query:: average gain of identical

In [17]:
query4 = (
    df
    .filter(
        (pl.col("change_type") == CHANGE_TYPES.identical.name) &
        (pl.col("node") == NODES.node1.name)
    )
    .groupby(['channel'])
    .agg(pl.col("gain").mean())
)
run_query(query4)

channel,gain
str,f64
"""Web_Click_Through_Rate_AGB""",62.106794
"""Mobile_Click_Through_Rate_AGB""",48.598743


#### query:: avg gain of prune to split

In [18]:
q1 = (
    df
    .filter(
        (pl.col("change_type") == CHANGE_TYPES.prune_to_split.name) &
        (pl.col("node") == NODES.node1.name) &
        (pl.col("tree_id") == "tree1")
        # (pl.col("snapshot") == )
    )
    .groupby(['channel'])
    .agg(pl.col("gain").mean())
)
q2 = (
    df
    .filter(
        (pl.col("change_type") == CHANGE_TYPES.prune_to_split.name) &
        (pl.col("node") == NODES.node2.name) &
        (pl.col("tree_id") == "tree1")
    )
    .groupby(['channel'])
    .agg(pl.col("gain").mean())
    .rename({
        "gain": "gain_"
    })
)


run_query(q1.join(q2, on="channel"))

channel,gain,gain_
str,f64,f64
"""Mobile_Click_Through_Rate_AGB""",0.0,2.469273
"""Web_Click_Through_Rate_AGB""",0.0,2.429723


#### query:: split to split

In [19]:
q11 = (
    df
    .filter(
        (pl.col("change_type") == CHANGE_TYPES.split_to_split.name) &
        (pl.col("node") == NODES.node1.name) &
        (pl.col("tree_id") == "tree1")
    )
    .groupby(['channel', 'node'])
    .agg(pl.col("gain").mean())
)
q12 = (
    df
    .filter(
        (pl.col("change_type") == CHANGE_TYPES.split_to_split.name) &
        (pl.col("node") == NODES.node2.name) &
        (pl.col("tree_id") == "tree1")
    )
    .groupby(['channel', "node"])
    .agg(pl.col("gain").mean())
    .rename({
        "gain": "gain_"
    })
)


run_query(q11.join(q12, on="channel"))

channel,node,gain,node_right,gain_
str,str,f64,str,f64
"""Web_Click_Through_Rate_AGB""","""node1""",13.883167,"""node2""",20.762109
"""Mobile_Click_Through_Rate_AGB""","""node1""",111.090613,"""node2""",47.243708


In [20]:
qq = (
    df
    .filter(
        (pl.col("change_type") == CHANGE_TYPES.split_to_split.name) &
        (pl.col("tree_id") == "tree1") &
        (pl.col("channel") == "Mobile_Click_Through_Rate_AGB")
    )
    .sort(["snapshot", "depth", "node"])
)
run_query(qq)

channel,snapshot_from,snapshot_to,snapshot,tree_id,change_type,node,predictor,score,parent_node,gain,split,left_child,right_child,depth,flag
str,str,str,str,str,str,str,str,f64,f64,f64,str,f64,f64,i64,str
"""Mobile_Click_Through_Rate_AGB""","""20230303""","""20230304""","""20230303""","""tree1""","""split_to_split""","""node1""","""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.195688,null,330.958696,"""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount < 33.0""",2.0,7.0,1,"""goldenrod"""
"""Mobile_Click_Through_Rate_AGB""","""20230303""","""20230304""","""20230303""","""tree1""","""split_to_split""","""node2""","""Param.DayPartBrowsing""",-0.186647,null,107.764421,"""Param.DayPartBrowsing in { night, evening }""",2.0,5.0,1,"""goldenrod"""
"""Mobile_Click_Through_Rate_AGB""","""20230304""","""20230305""","""20230304""","""tree1""","""split_to_split""","""node1""","""Param.DayPartBrowsing""",-0.186647,null,107.764421,"""Param.DayPartBrowsing in { night, evening }""",2.0,5.0,1,"""goldenrod"""
"""Mobile_Click_Through_Rate_AGB""","""20230304""","""20230305""","""20230304""","""tree1""","""split_to_split""","""node2""","""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.19303,null,75.266775,"""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount < 100.0""",2.0,13.0,1,"""goldenrod"""
"""Mobile_Click_Through_Rate_AGB""","""20230305""","""20230306""","""20230305""","""tree1""","""split_to_split""","""node1""","""Customer.OrderShareL1Speelgoed""",-0.147841,13.0,2.502198,"""Customer.OrderShareL1Speelgoed < 0.012""",15.0,16.0,3,"""goldenrod"""
"""Mobile_Click_Through_Rate_AGB""","""20230305""","""20230306""","""20230305""","""tree1""","""split_to_split""","""node2""","""Customer.OrderShareL1KantoorSchool""",-0.149421,19.0,3.6817,"""Customer.OrderShareL1KantoorSchool < 0.063""",21.0,22.0,3,"""goldenrod"""
"""Mobile_Click_Through_Rate_AGB""","""20230306""","""20230307""","""20230306""","""tree1""","""split_to_split""","""node1""","""pyName""",-0.188131,3.0,3.137135,"""pyName in { LivingNewCollection, Elektronicatvhomeaudioandhifi, DrogisterijDeals, Sport, Pregnant, …",12.0,15.0,4,"""goldenrod"""
"""Mobile_Click_Through_Rate_AGB""","""20230306""","""20230307""","""20230306""","""tree1""","""split_to_split""","""node2""","""Param.ClickedCount30Days""",-0.188436,3.0,2.261935,"""Param.ClickedCount30Days < 2.0""",18.0,19.0,4,"""goldenrod"""


In [23]:
qq = (
    df
    .filter(
        (pl.col("change_type") == CHANGE_TYPES.split_to_split.name) &
        (pl.col("tree_id") == "tree1") &
        (pl.col("channel") == "Web_Click_Through_Rate_AGB")
    )
    .sort(["snapshot", "depth", "node"])
)
run_query(qq)

channel,snapshot_from,snapshot_to,snapshot,tree_id,change_type,node,predictor,score,parent_node,gain,split,left_child,right_child,depth,flag
str,str,str,str,str,str,str,str,f64,f64,f64,str,f64,f64,i64,str
"""Web_Click_Through_Rate_AGB""","""20230303""","""20230304""","""20230303""","""tree1""","""split_to_split""","""node1""","""Customer.OrderShareL1Beauty""",-0.179706,1.0,14.361981,"""Customer.OrderShareL1Beauty < 0.25""",20.0,23.0,2,"""goldenrod"""
"""Web_Click_Through_Rate_AGB""","""20230303""","""20230304""","""20230303""","""tree1""","""split_to_split""","""node2""","""pyIssue""",-0.186736,1.0,49.982383,"""pyIssue in { PaidAdvertisements }""",20.0,29.0,2,"""goldenrod"""
"""Web_Click_Through_Rate_AGB""","""20230303""","""20230304""","""20230303""","""tree1""","""split_to_split""","""node1""","""IH.Web.Inbound.Clicked.pxLastGroupID""",-0.197262,2.0,9.360646,"""IH.Web.Inbound.Clicked.pxLastGroupID is Missing""",4.0,7.0,3,"""goldenrod"""
"""Web_Click_Through_Rate_AGB""","""20230303""","""20230304""","""20230303""","""tree1""","""split_to_split""","""node2""","""Param.ShelfLevel2""",-0.198251,2.0,33.74027,"""Param.ShelfLevel2 in { 21357, 10714, 12525, 47536, 10494, 54483, 25152, 1701, 24421, 10596, 12885, …",4.0,5.0,3,"""goldenrod"""
"""Web_Click_Through_Rate_AGB""","""20230304""","""20230305""","""20230304""","""tree1""","""split_to_split""","""node1""","""Customer.BehavioralData.VisitShareL1Persoonlijkeverzorging""",-0.18524,10.0,0.330966,"""Customer.BehavioralData.VisitShareL1Persoonlijkeverzorging < 0.102""",12.0,13.0,4,"""goldenrod"""
"""Web_Click_Through_Rate_AGB""","""20230304""","""20230305""","""20230304""","""tree1""","""split_to_split""","""node2""","""Param.ClickedCount1Day""",-0.185754,14.0,0.369608,"""Param.ClickedCount1Day < 0.0""",16.0,17.0,4,"""goldenrod"""
"""Web_Click_Through_Rate_AGB""","""20230306""","""20230307""","""20230306""","""tree1""","""split_to_split""","""node1""","""Param.ShelfLevel2""",-0.197275,2.0,34.743149,"""Param.ShelfLevel2 in { 21357, 10714, 12525, 47536, 10494, 54483, 25152, 1701, 24421, 10596, 12885, …",4.0,9.0,3,"""goldenrod"""
"""Web_Click_Through_Rate_AGB""","""20230306""","""20230307""","""20230306""","""tree1""","""split_to_split""","""node2""","""pyName""",-0.197612,2.0,7.658323,"""pyName in { AVBPlayStation5 }""",4.0,5.0,3,"""goldenrod"""
"""Web_Click_Through_Rate_AGB""","""20230307""","""20230308""","""20230307""","""tree1""","""split_to_split""","""node1""","""IH.Web.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.185224,2.0,10.619092,"""IH.Web.Inbound.Clicked.pyHistoricalOutcomeCount < 2.0""",7.0,12.0,3,"""goldenrod"""


In [24]:
qq1 = (
    df
    .filter(
        # (pl.col("change_type") == CHANGE_TYPES.split_to_split.name) &
        (pl.col("tree_id") == "tree1") &
        (pl.col("channel") == "Mobile_Click_Through_Rate_AGB") &
        (pl.col("depth") == 1)
    )
    .sort(["snapshot", "depth", "node"])
)
run_query(qq1)

channel,snapshot_from,snapshot_to,snapshot,tree_id,change_type,node,predictor,score,parent_node,gain,split,left_child,right_child,depth,flag
str,str,str,str,str,str,str,str,f64,f64,f64,str,f64,f64,i64,str
"""Mobile_Click_Through_Rate_AGB""","""20230303""","""20230304""","""20230303""","""tree1""","""split_to_split""","""node1""","""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.195688,null,330.958696,"""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount < 33.0""",2.0,7.0,1,"""goldenrod"""
"""Mobile_Click_Through_Rate_AGB""","""20230303""","""20230304""","""20230303""","""tree1""","""split_to_split""","""node2""","""Param.DayPartBrowsing""",-0.186647,null,107.764421,"""Param.DayPartBrowsing in { night, evening }""",2.0,5.0,1,"""goldenrod"""
"""Mobile_Click_Through_Rate_AGB""","""20230304""","""20230305""","""20230304""","""tree1""","""split_to_split""","""node1""","""Param.DayPartBrowsing""",-0.186647,null,107.764421,"""Param.DayPartBrowsing in { night, evening }""",2.0,5.0,1,"""goldenrod"""
"""Mobile_Click_Through_Rate_AGB""","""20230304""","""20230305""","""20230304""","""tree1""","""split_to_split""","""node2""","""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.19303,null,75.266775,"""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount < 100.0""",2.0,13.0,1,"""goldenrod"""
"""Mobile_Click_Through_Rate_AGB""","""20230305""","""20230306""","""20230305""","""tree1""","""identical""","""node1""","""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.19303,null,75.266775,"""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount < 100.0""",2.0,13.0,1,"""white"""
"""Mobile_Click_Through_Rate_AGB""","""20230305""","""20230306""","""20230305""","""tree1""","""identical""","""node2""","""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.191876,null,130.076381,"""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount < 96.0""",2.0,19.0,1,"""white"""
"""Mobile_Click_Through_Rate_AGB""","""20230306""","""20230307""","""20230306""","""tree1""","""identical""","""node1""","""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.191876,null,130.076381,"""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount < 96.0""",2.0,19.0,1,"""white"""
"""Mobile_Click_Through_Rate_AGB""","""20230306""","""20230307""","""20230306""","""tree1""","""identical""","""node2""","""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.194388,null,112.757158,"""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount < 96.0""",2.0,21.0,1,"""white"""
"""Mobile_Click_Through_Rate_AGB""","""20230307""","""20230308""","""20230307""","""tree1""","""identical""","""node1""","""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.194388,null,112.757158,"""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount < 96.0""",2.0,21.0,1,"""white"""


In [35]:
qq1 = (
    df
    .filter(
        # (pl.col("change_type") == CHANGE_TYPES.split_to_split.name) &
        (pl.col("tree_id") == "tree1") &
        (pl.col("channel") == "Mobile_Click_Through_Rate_AGB") &
        (pl.col("depth") == 1)
    )
    .sort(["snapshot", "depth", "node"])
)
run_query(qq1)

channel,snapshot_from,snapshot_to,snapshot,tree_id,change_type,node,predictor,score,parent_node,gain,split,left_child,right_child,depth,flag
str,str,str,str,str,str,str,str,f64,f64,f64,str,f64,f64,i64,str
"""Mobile_Click_Through_Rate_AGB""","""20230303""","""20230304""","""20230303""","""tree1""","""split_to_split""","""node1""","""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.195688,null,330.958696,"""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount < 33.0""",2.0,7.0,1,"""goldenrod"""
"""Mobile_Click_Through_Rate_AGB""","""20230303""","""20230304""","""20230303""","""tree1""","""split_to_split""","""node2""","""Param.DayPartBrowsing""",-0.186647,null,107.764421,"""Param.DayPartBrowsing in { night, evening }""",2.0,5.0,1,"""goldenrod"""
"""Mobile_Click_Through_Rate_AGB""","""20230304""","""20230305""","""20230304""","""tree1""","""split_to_split""","""node1""","""Param.DayPartBrowsing""",-0.186647,null,107.764421,"""Param.DayPartBrowsing in { night, evening }""",2.0,5.0,1,"""goldenrod"""
"""Mobile_Click_Through_Rate_AGB""","""20230304""","""20230305""","""20230304""","""tree1""","""split_to_split""","""node2""","""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.19303,null,75.266775,"""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount < 100.0""",2.0,13.0,1,"""goldenrod"""
"""Mobile_Click_Through_Rate_AGB""","""20230305""","""20230306""","""20230305""","""tree1""","""identical""","""node1""","""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.19303,null,75.266775,"""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount < 100.0""",2.0,13.0,1,"""white"""
"""Mobile_Click_Through_Rate_AGB""","""20230305""","""20230306""","""20230305""","""tree1""","""identical""","""node2""","""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.191876,null,130.076381,"""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount < 96.0""",2.0,19.0,1,"""white"""
"""Mobile_Click_Through_Rate_AGB""","""20230306""","""20230307""","""20230306""","""tree1""","""identical""","""node1""","""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.191876,null,130.076381,"""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount < 96.0""",2.0,19.0,1,"""white"""
"""Mobile_Click_Through_Rate_AGB""","""20230306""","""20230307""","""20230306""","""tree1""","""identical""","""node2""","""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.194388,null,112.757158,"""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount < 96.0""",2.0,21.0,1,"""white"""
"""Mobile_Click_Through_Rate_AGB""","""20230307""","""20230308""","""20230307""","""tree1""","""identical""","""node1""","""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount""",-0.194388,null,112.757158,"""IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount < 96.0""",2.0,21.0,1,"""white"""
